# NB01 - Data Ingestion avec CTAS et COPY INTO

## A. Mise en place
Le notebook dans `Resources/NB01/Setup` prend les fichier **.parquet** et les copie dans le volume `demo_<username>.demo_lakeflow_connect.lfc_demo`.


In [0]:
%run "./Resources/NB01/Setup"

In [0]:
%sql
select current_catalog(), current_schema();

## B. Fichiers sources

1. On va créer une table contenant les données historiques des commandes sur 2025, à partir des fichiers **parquet** présent dans le volume **/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo**.

On peut vérifier les fichiers disponibles en utilisant la fonction `LIST` :

In [0]:
df = spark.sql(f"""
          LIST '/Volumes/{catalog_name}/{schema}/{volume}'
          """)
display(df)


In [0]:
df = spark.sql(f"""
          select * from parquet.`/Volumes/{catalog_name}/{schema}/{volume}/ventes_2025_01.parquet`
          """)
display(df)

## C. Ingéstion des données en mode batch avec CTAS et read_files()

CTAS ou `CREATE TABLE AS` permet de créer et d'alimenter une table avec le resultats d'une requête. Permet donc de créer et de charger une table en une seule étape.

### Inférence automatique de schéma avec les fichiers Parquet

Les fichiers Parquet contiennent des métadonnées décrivant le schéma des données (types, colonnes, etc.). Lors de la lecture d'un fichier Parquet, Spark ou Databricks détecte automatiquement la structure des données sans avoir besoin de spécifier le schéma manuellement. Cela simplifie l'ingestion, car le schéma est directement extrait du fichier, garantissant une correspondance exacte avec les données stockées.

### C1. CTAS avec la fonction read_files() : 

[read_files() documentation](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)

In [0]:
df = spark.sql(f"""
    SELECT *
    FROM read_files(
        '/Volumes/{catalog_name}/{schema}/{volume}',
        format => 'parquet'
        )
    LIMIT 10
""")
display(df)

In [0]:
%sql
DROP TABLE IF EXISTS historical_sales_data_bronze;

In [0]:
spark.sql(f"""
    CREATE TABLE historical_sales_data_bronze
    SELECT *
    FROM read_files(
        '/Volumes/{catalog_name}/{schema}/{volume}',
        format => 'parquet'
        )
    """
)

display(spark.sql("SELECT * FROM historical_sales_data_bronze"))

In [0]:
%sql
DESCRIBE TABLE EXTENDED historical_sales_data_bronze;

La requête `DESCRIBE TABLE EXTENDED historical_sales_data_bronze` affiche des informations détaillées sur la table `historical_sales_data_bronze`. Elle liste les colonnes, leurs types, et fournit des métadonnées supplémentaires comme le format de stockage, le chemin du fichier, les propriétés de la table, et d'autres informations techniques utiles pour comprendre la structure et l'emplacement des données.

### C2. Ingestion avec Python

Jusque là on utilisait des cellules python et `spark.sql()` pour exécuter les requêtes pour pouvoir utiliser les variables : 
- `catalog_name`
- `schema`
- `volume`
mais ces requêtes fonctionneraient aussi dans des cellules SQL en remplacant ces variables.

Maintenant on va faire la même chose en python : 

In [0]:
df = (spark
      .read
      .format("parquet")
      .load(f"/Volumes/{catalog_name}/{schema}/{volume}")
)

(df
 .write
 .mode("overwrite")
 .saveAsTable(f"{catalog_name}.{schema}.historical_sales_data_bronze_python")
)

sales_bronze_table = spark.table(f"{catalog_name}.{schema}.historical_sales_data_bronze_python")
sales_bronze_table.display()

## D. Ingéstion incrémentale avec `COPY INTO`

La fonction `COPY INTO` permet d'ingérer des données de manière incrémentale dans une table à partir de fichiers externes (parquet, csv, etc.). Elle détecte automatiquement les nouveaux fichiers ou les fichiers modifiés et ne copie que ceux-ci, évitant les doublons. Cette commande est idéale pour charger régulièrement des données sans recharger l'ensemble des fichiers à chaque fois. Elle prend en charge l'inférence de schéma, la gestion des erreurs, et peut être utilisée avec différents formats de fichiers.

Exemple :
```sql
COPY INTO ma_table
FROM '/Volumes/demo_{username}/demo_lakeflow_connect/lfc_demo/'
FILEFORMAT = PARQUET;
```

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ventes (
 
    -- Identifiants
    commande_id         VARCHAR(20)     NOT NULL,
    client_id           VARCHAR(10)     NOT NULL,
 
    -- Informations client
    client_nom          VARCHAR(100)    NOT NULL,
    client_prenom       VARCHAR(100)    NOT NULL,
    client_email        VARCHAR(255)    NOT NULL,
    client_region       VARCHAR(100)    NOT NULL,
 
    -- Informations produit
    produit_id          VARCHAR(10)     NOT NULL,
    produit_nom         VARCHAR(150)    NOT NULL,
    produit_categorie   VARCHAR(100)    NOT NULL,
 
    -- Détails commande
    quantite            INTEGER         NOT NULL,
    prix_unitaire       NUMERIC(10, 2)  NOT NULL,
    remise              DECIMAL(12, 2)   NOT NULL,
    montant_ht          NUMERIC(12, 2)  NOT NULL,
    tva                 NUMERIC(12, 2)  NOT NULL,
    montant_ttc         NUMERIC(12, 2)  NOT NULL,
 
    -- Statut et paiement
    statut              VARCHAR(50)     NOT NULL,
    mode_paiement       VARCHAR(50)     NOT NULL,
 
    -- Dates
    date_commande       DATE            NOT NULL,
    date_livraison      DATE            NOT NULL,
 
    -- Clé primaire
    CONSTRAINT pk_ventes PRIMARY KEY (commande_id)
);

-- 2. Activer la feature
ALTER TABLE ventes SET TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

-- 3. Ajouter le DEFAULT
ALTER TABLE ventes ALTER COLUMN remise SET DEFAULT 0.00;

In [0]:
%sql
COPY INTO ventes
FROM (
    SELECT 
        commande_id,
        client_id,
        client_nom,
        client_prenom,
        client_email,
        client_region,
        produit_id,
        produit_nom,
        produit_categorie,
        quantite,
        CAST(prix_unitaire AS DECIMAL(10, 2)) AS prix_unitaire,
        CAST(remise AS DECIMAL(5,2)) AS remise,
        CAST(montant_ht AS DECIMAL(12, 2)) AS montant_ht,
        CAST(tva AS DECIMAL(12, 2)) AS tva,
        CAST(montant_ttc AS DECIMAL(12, 2)) AS montant_ttc,
        statut,
        mode_paiement,
        CAST(date_commande AS DATE) AS date_commande,
        CAST(date_livraison AS DATE) AS date_livraison
    FROM '/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo/'
)
FILEFORMAT = PARQUET;

### Erreur : Common Schema Mismatch Error
L'erreur **Common Schema Mismatch Error** survient lorsque le schéma des données sources (par exemple, les fichiers Parquet) ne correspond pas exactement au schéma de la table cible dans laquelle on tente d'insérer les données. Cela peut concerner le nom, le type ou l'ordre des colonnes. Pour éviter cette erreur, il faut s'assurer que les colonnes sélectionnées dans la requête source correspondent exactement à la définition de la table cible, notamment en utilisant des `CAST` pour harmoniser les types si besoin.

**Exemple** : on va créer exactement le même schéma que le précédent en retirant la colonne *client_nom* pour simuler l'erreur :

In [0]:
%sql
DROP TABLE IF EXISTS ventes_error;

CREATE TABLE ventes_error AS
SELECT * FROM read_files(
    '/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo/',
    format => 'parquet'
)
LIMIT 0;

ALTER TABLE ventes_error SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE ventes_error DROP COLUMN client_nom;

In [0]:
%sql
COPY INTO ventes_error
FROM '/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo/'
FILEFORMAT = PARQUET;

### Gestion des erreurs avec `COPY_OPTIONS`
On peut rajouter dans la requête `COPY_OPTIONS ('mergeSchema' = 'true')`

In [0]:
%sql
COPY INTO ventes_error
FROM '/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo/'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
-- On voit bien la colonne supplémentaire à la fin de la table
select * from ventes_error
LIMIT 10;

### Prévoir l'évolution de schéma

On peut créer dans un premier temps une table vide (sans schéma) et ensuite utiliser le `COPY_OPTIONS`.

Pour anticiper l'évolution du schéma lors de l'ingestion de données (par exemple, ajout de colonnes dans les fichiers sources), utilisez l'option `COPY_OPTIONS ('mergeSchema' = 'true')` avec la commande `COPY INTO`. Cela permet à Databricks d'ajuster automatiquement la table cible pour intégrer les nouvelles colonnes détectées dans les fichiers sources.

**Exemple :**
```sql
COPY INTO ma_table
FROM '/Volumes/mon_catalogue/mon_schema/mon_volume/'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');
```

In [0]:
%sql
DROP TABLE IF EXISTS ventes_no_schema;

CREATE TABLE ventes_no_schema;

COPY INTO ventes_no_schema
FROM '/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo/'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');


### Principe d'Idempotence pour l'Ingestion Incrémentale

L'idempotence est un principe clé lors de l'ingestion incrémentale de données. Une opération est dite *idempotente* si, lorsqu'elle est exécutée plusieurs fois, le résultat reste le même que si elle n'avait été exécutée qu'une seule fois. 

Dans le contexte de l'ingestion incrémentale, cela signifie que rejouer plusieurs fois le même lot de données n'entraîne pas de doublons ni de modifications inattendues dans la table cible. Cela garantit la fiabilité et la cohérence des données, même en cas de relance d'un job ou d'une panne.

Pour assurer l'idempotence, il est courant d'utiliser des clés uniques (par exemple, un identifiant de commande ou un horodatage) et des opérations comme `MERGE` ou `COPY INTO` avec des options permettant d'éviter les doublons.

Par exemple si on réexcute la requête suivante on ne devrait pas avoir de nouvelles lignes insérées dans la table car les fichiers ont déjà été traités.

In [0]:
%sql
COPY INTO ventes_no_schema
FROM '/Volumes/demo_edadou/demo_lakeflow_connect/lfc_demo/'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');